In [164]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np
import pandas as pd
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter
import re

In [165]:
with open('persuasion.txt') as f:
    content = f.read()
print(len(content))
content[:100]

484132


'The Project Gutenberg eBook of Persuasion\n    \nThis eBook is for the use of anyone anywhere in the U'

In [166]:
import tiktoken

enc = tiktoken.get_encoding('gpt2')
vocab_size = enc.n_vocab
vocab_size

50257

In [167]:
def tokenise_text(text):
    tokens = enc.encode(text)
    return tokens
encoded_text = tokenise_text(content)

# vocab_size = len(set(encoded_text))
encoded_text = torch.tensor(encoded_text)
print(vocab_size)

50257


In [168]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)

mps


In [169]:
torch.random.manual_seed(1234)

In [170]:
context_length = 16
embedding_dim = 512


class Head(nn.Module):
    def __init__(self, head_dimension, n_embd):
        super().__init__()

        self.query = nn.Linear(n_embd, head_dimension, bias=False)
        self.key = nn.Linear(n_embd, head_dimension, bias=False)
        self.value = nn.Linear(n_embd, head_dimension, bias=False)
        
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
        

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1)*(C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        out =  wei @ v

        return out
    
    
class TransformerNameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd):
        super().__init__()
        self.n_embd = n_embd
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(context_length, n_embd)
        self.head_model = Head(self.n_embd, self.n_embd)
        self.llm_model = nn.Linear(n_embd, vocab_size)
            

    def forward(self, idx):
        B, T = idx.shape
 
        token_embedding = self.token_embedding(idx) # B, T, embd_n
 
        pos = torch.arange(T)
 
        positional_embeddings = self.position_embedding(pos) # T, nemb
 
        x = token_embedding + positional_embeddings

        x = self.head_model(x) # B, T, n_embd
        logits = self.llm_model(x)
 
        return logits


In [171]:
model = TransformerNameGenerator(vocab_size, context_length, embedding_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [172]:
from torch.utils.data import TensorDataset, DataLoader

xs = []
ys = []
for i in range(0, len(encoded_text)-context_length):
    x_chunk = encoded_text[i:i+context_length]
    y_chunk = encoded_text[i+1:i+context_length+1]

    xs.append(x_chunk)
    ys.append(y_chunk)

X = torch.stack(xs)
Y = torch.stack(ys)

dataset = TensorDataset(X, Y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
writer = SummaryWriter()

for epoch in range(1):
    for i, (xb, yb) in enumerate(loader):
        logits = model.forward(xb)
        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = yb.view(B*T)
        loss = F.cross_entropy(logits, targets)
        writer.add_scalar("Loss/train", loss, i)
        if not i%10:
            print(loss.item(), f'Epoch: {epoch}, Batch: {i}')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

10.853995323181152 Epoch: 0, Batch: 0
11.110276222229004 Epoch: 0, Batch: 10
10.23242473602295 Epoch: 0, Batch: 20
8.859140396118164 Epoch: 0, Batch: 30
8.19334602355957 Epoch: 0, Batch: 40
8.58470630645752 Epoch: 0, Batch: 50
8.149955749511719 Epoch: 0, Batch: 60
8.41510009765625 Epoch: 0, Batch: 70
8.331182479858398 Epoch: 0, Batch: 80
7.935187339782715 Epoch: 0, Batch: 90
8.096390724182129 Epoch: 0, Batch: 100
7.8750762939453125 Epoch: 0, Batch: 110
8.170085906982422 Epoch: 0, Batch: 120
8.844783782958984 Epoch: 0, Batch: 130
10.584354400634766 Epoch: 0, Batch: 140
12.823541641235352 Epoch: 0, Batch: 150
14.9750394821167 Epoch: 0, Batch: 160
11.978150367736816 Epoch: 0, Batch: 170
13.23440933227539 Epoch: 0, Batch: 180
13.085479736328125 Epoch: 0, Batch: 190
12.606083869934082 Epoch: 0, Batch: 200
13.132455825805664 Epoch: 0, Batch: 210
19.460296630859375 Epoch: 0, Batch: 220
19.876070022583008 Epoch: 0, Batch: 230
16.677261352539062 Epoch: 0, Batch: 240
15.33694839477539 Epoch: 0, 

In [ ]:
for xb, yb in loader:
    print(xb.shape)
    break

torch.Size([32, 16])


In [ ]:
def generate(max_words, prompt_text):
    prompt_tokens = tokenise_text(prompt_text)
    
    idx = torch.tensor([[enc.decode(w) for w in prompt_tokens]], dtype=torch.long)
    
    for _ in range(max_words):
        idx_cond = idx[:, -context_length:]
        logits = model(idx_cond)

        logits = logits[:, -1, :]

        probs = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_idx), dim=1)

    words = [enc.decode(i) for i in idx[0].tolist()]
    return ' '.join(words)

generate(16, "Tell me a story about ")

'tell me a story about . chapter chapter chapter chapter chapter solemnity chapter were chapter but chapter gutenberg held chapter held'